# 01 – Descoberta dos microdados do TIC Kids Online Brasil 2025
Projeto: *Uso de IA Generativa para Fins Escolares entre Crianças e Adolescentes de 9 a 14 Anos* (IA – 7º CC, Mackenzie)

**Objetivo deste notebook:** ler a base de microdados do Cetic.br, listar as colunas e localizar as variáveis relevantes (idade, região, classe, pesos amostrais, uso de IA generativa), para definir a variável-alvo com base no dicionário. Ele **não** treina modelos.

**Como usar**
1. Baixe os microdados em https://cetic.br/en/microdados/ (TIC Kids Online Brasil 2025) e o dicionário de variáveis.
2. No Colab, envie o arquivo da base (CSV, SAV ou XLSX) para `/content` (ícone de pasta > upload) e ajuste `ARQUIVO` abaixo.
3. Execute tudo e cole a saída das duas últimas células na conversa.

In [ ]:
# =============================================================================
# Projeto: Uso de IA Generativa para Fins Escolares (9 a 14 anos) - TIC Kids Online Brasil 2025
# Arquivo: 01_descoberta_tic_kids_2025.ipynb
# Integrante: Kauã Paixão - RA 10418548
# Síntese: leitura dos microdados, listagem de colunas e busca de variáveis relevantes.
# Histórico de alterações:
#   2026-11-19 | Kauã Paixão | Criação do notebook de descoberta da base.
# =============================================================================
import os, re, glob
import pandas as pd

ARQUIVO = ""   # ex.: "/content/tic_kids_online_2025_criancas.csv" (deixe vazio para listar os arquivos de /content)
if not ARQUIVO:
    print("Arquivos encontrados em /content:")
    for f in sorted(glob.glob("/content/*.*")): print(" -", f)

In [ ]:
def ler_base(caminho):
    """Lê CSV, SAV (SPSS) ou XLSX, tentando separadores e codificações comuns."""
    ext = os.path.splitext(caminho)[1].lower()
    if ext == ".csv":
        for enc in ("utf-8", "latin-1"):
            for sep in (";", ",", "\t", "|"):
                try:
                    df = pd.read_csv(caminho, sep=sep, encoding=enc, low_memory=False)
                    if df.shape[1] > 1:
                        print(f"CSV lido com sep={sep!r} e encoding={enc}")
                        return df
                except Exception:
                    continue
        raise ValueError("Não consegui ler o CSV: informe o separador e a codificação.")
    if ext == ".sav":
        try:
            import pyreadstat
        except ImportError:
            raise ImportError("Execute em uma célula: !pip install pyreadstat")
        df, meta = pyreadstat.read_sav(caminho)
        return df
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(caminho)
    raise ValueError(f"Formato não suportado: {ext}")

df = ler_base(ARQUIVO)
print("Registros:", df.shape[0], "| Variáveis:", df.shape[1])

In [ ]:
# Busca de variáveis relevantes pelo nome da coluna (palavras-chave; ajuste se necessário)
palavras = ["ia", "intelig", "chatgpt", "gemini", "copilot", "escola", "estud", "pesquis", "idade", "regiao",
            "região", "sexo", "classe", "area", "área", "peso", "pond", "internet", "habil"]
cols = list(df.columns)
achadas = {p: [c for c in cols if p in str(c).lower()] for p in palavras}
for p, lista in achadas.items():
    if lista: print(f"{p:10s} ({len(lista):3d}): {lista[:15]}")

In [ ]:
# SAÍDA PARA COLAR NA CONVERSA: lista completa de colunas, tipos e nº de valores distintos
resumo = pd.DataFrame({"coluna": cols,
                       "tipo": [str(df[c].dtype) for c in cols],
                       "distintos": [df[c].nunique(dropna=True) for c in cols],
                       "ausentes": [int(df[c].isna().sum()) for c in cols]})
print(resumo.to_string(index=False))
resumo.to_csv("colunas_tic_kids_2025.csv", index=False)
print("\nArquivo salvo: colunas_tic_kids_2025.csv")